# EDMS Schema

**Author**: Advay Bhagwat  
**Email**: advay.bhagwat@gmail.com  
**Date**: October 19, 2025  

---

## `endpoints` Table

This table is the primary table for endpoint information.

```sql
CREATE TABLE endpoints (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    endpoint_id TEXT UNIQUE NOT NULL,           -- E001, E002, etc.
    endpoint_str TEXT NOT NULL,                 -- "/api/v1/users"
    annotation TEXT,                            -- Markdown-constrained annotation
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE INDEX idx_endpoints_id ON endpoints(endpoint_id);
CREATE INDEX idx_endpoints_str ON endpoints(endpoint_str);
```

**Notes**:
- `endpoint_id` is the unique identifier for cross-referencing
- `endpoint_str` is the actual endpoint path
- `annotation` allows one-liner documentation
- Timestamps for tracking and sorting

---

## `request_metadata` Table

This table stores metadata about requests. Full request data (including headers) is stored in JSON files on disk.

```sql
CREATE TABLE request_metadata (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    endpoint_id TEXT NOT NULL,                  -- Reference (no FK)
    request_number INTEGER NOT NULL,            -- Maps to file: request-NNN.json
    file_path TEXT NOT NULL,                    -- Relative path to JSON file
    method TEXT,                                -- GET, POST, PUT, DELETE, etc.
    timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE INDEX idx_request_endpoint ON request_metadata(endpoint_id);
CREATE INDEX idx_request_timestamp ON request_metadata(timestamp);
CREATE INDEX idx_request_method ON request_metadata(method);
```

**Notes**:
- **No Foreign Keys** - references `endpoint_id` but not enforced
- `request_number` maps to JSON filename (e.g., 1 → request-001.json)
- `file_path` stores relative path (e.g., "E001/request-001.json")
- `method` stored as column for queryable filtering
- Full request body and headers stored in JSON file on disk
- Application-level joins used instead of database constraints

---

## `response_metadata` Table

This table stores metadata about responses. Full response data (including headers) is stored in JSON files on disk.

```sql
CREATE TABLE response_metadata (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    endpoint_id TEXT NOT NULL,                  -- Reference (no FK)
    request_number INTEGER NOT NULL,            -- Maps to matching request
    file_path TEXT NOT NULL,                    -- Relative path to JSON file
    status_code INTEGER,                        -- HTTP status: 200, 404, 500, etc.
    exit_code INTEGER,                          -- Process exit code
    response_time_ms INTEGER,                   -- Response time in milliseconds
    timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE INDEX idx_response_endpoint ON response_metadata(endpoint_id);
CREATE INDEX idx_response_status ON response_metadata(status_code);
CREATE INDEX idx_response_timestamp ON response_metadata(timestamp);
```

**Notes**:
- **No Foreign Keys** - references `endpoint_id` but not enforced
- `request_number` maps to matching request (same index)
- `file_path` stores relative path (e.g., "E001/response-001.json")
- `status_code` stored as column for queryable filtering (e.g., find all 404s)
- `exit_code` for process-level exit codes
- `response_time_ms` optional performance tracking
- Full response body and headers stored in JSON file on disk
- Application-level joins used to link with requests

---

## `metadata` Table

This table contains analytics and aggregated data per endpoint.

```sql
CREATE TABLE metadata (
    endpoint_id TEXT PRIMARY KEY,               -- One row per endpoint (no FK)
    request_count INTEGER DEFAULT 0,
    response_count INTEGER DEFAULT 0,
    data_size_bytes INTEGER DEFAULT 0,          -- Total data size on disk
    last_tested TIMESTAMP,
    avg_response_time_ms INTEGER                -- Average response time
);
```

**Notes**:
- **No Foreign Keys** - references `endpoint_id` but not enforced
- Denormalized counts for quick access (avoids COUNT queries)
- `data_size_bytes` tracks total storage on disk
- `last_tested` for sorting by recency
- `avg_response_time_ms` optional performance analytics
- Updated via application logic (not triggers)

---

## `tags` Table

This table implements endpoint tagging without foreign key constraints.

```sql
CREATE TABLE tags (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    endpoint_id TEXT NOT NULL,                  -- Reference (no FK)
    tag TEXT NOT NULL,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    UNIQUE(endpoint_id, tag)                    -- Prevent duplicate tags
);

CREATE INDEX idx_tags_endpoint ON tags(endpoint_id);
CREATE INDEX idx_tags_tag ON tags(tag);
```

**Notes**:
- **No Foreign Keys** - references `endpoint_id` but not enforced
- Simple structure for tagging system
- UNIQUE constraint enforces no duplicate tags per endpoint
- Indexed on both `endpoint_id` and `tag` for fast queries
- Application layer enforces 25-tag limit (planned for v1.1)

---

## `bookmarks` Table

This table manages saved endpoints organized in folders.

```sql
CREATE TABLE bookmarks (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    endpoint_id TEXT NOT NULL,                  -- Reference (no FK)
    folder TEXT,                                -- NULL for root level
    notes TEXT,                                 -- Additional bookmark notes
    timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE INDEX idx_bookmarks_endpoint ON bookmarks(endpoint_id);
CREATE INDEX idx_bookmarks_folder ON bookmarks(folder);
```

**Notes**:
- **No Foreign Keys** - references `endpoint_id` but not enforced
- Allows organizing bookmarked endpoints into folders
- `folder` can be NULL for unorganized bookmarks
- `notes` for additional context beyond endpoint annotation

---

## `history` Table

This table tracks all actions on endpoints.

```sql
CREATE TABLE history (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    endpoint_id TEXT NOT NULL,                  -- Reference (no FK)
    action TEXT NOT NULL,                       -- "test", "save", "delete", etc.
    details TEXT,                               -- JSON with action details
    timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE INDEX idx_history_endpoint ON history(endpoint_id);
CREATE INDEX idx_history_timestamp ON history(timestamp);
CREATE INDEX idx_history_action ON history(action);
```

**Notes**:
- **No Foreign Keys** - references `endpoint_id` but not enforced
- Audit trail for all endpoint operations
- `action` field for filtering by action type
- `details` stores JSON for flexible action metadata
- Used to populate "History of all endpoints tested" view

---

## File Structure Integration

EDMS uses a hybrid storage approach: SQLite for queryable metadata, disk files for full data.

```
edms_data/
├── E001/
│   ├── request-001.json        # Full request with headers
│   ├── response-001.json       # Full response with headers
│   ├── request-002.json
│   └── response-002.json
├── E002/
│   ├── request-001.json
│   └── response-001.json
└── edms.db                     # SQLite: metadata only
```

### JSON File Format

Request file example (`request-001.json`):
```json
{
    "method": "GET",
    "url": "/api/v1/users",
    "headers": {
        "Authorization": "Bearer token",
        "Content-Type": "application/json"
    },
    "body": {}
}
```

Response file example (`response-001.json`):
```json
{
    "status": 200,
    "headers": {
        "Content-Type": "application/json",
        "Server": "nginx"
    },
    "body": {
        "users": [...]
    }
}
```

---

## Storage Strategy

- **SQLite** - Queryable metadata only (columns, not JSON blobs)
- **Disk** - Full request/response data including headers
- SQLite stores **relative paths** to JSON files
- Leverages RDBMS query optimization for metadata
- Merge-friendly: no FK constraints, simple file copying

---

## Query Patterns

### Application-Level Joins (No FK)

Since we don't use foreign keys, joins happen at the application level:

```python
# Step 1: Query endpoint
endpoint = query("SELECT * FROM endpoints WHERE endpoint_id = ?", "E001")

# Step 2: Query responses (separate query, no FK)
responses = query("SELECT * FROM response_metadata WHERE endpoint_id = ?", "E001")

# Step 3: Combine in application
endpoint_data = {
    "endpoint": endpoint,
    "responses": responses
}
```

### Find All 404 Responses

```sql
SELECT 
    endpoint_id,
    file_path,
    status_code,
    timestamp
FROM response_metadata
WHERE status_code = 404
ORDER BY timestamp DESC;
```

### Count Errors Per Endpoint

```sql
SELECT 
    endpoint_id,
    COUNT(*) as error_count
FROM response_metadata
WHERE status_code >= 400
GROUP BY endpoint_id
ORDER BY error_count DESC;
```

### Get Endpoint with Tags

```sql
-- Query 1: Get endpoint
SELECT * FROM endpoints WHERE endpoint_id = 'E001';

-- Query 2: Get tags (application-level join)
SELECT tag FROM tags WHERE endpoint_id = 'E001';
```

### Recent Test Activity

```sql
SELECT 
    endpoint_id,
    action,
    timestamp
FROM history
WHERE action = 'test'
ORDER BY timestamp DESC
LIMIT 20;
```

---

## Design Principles

### Database Level

- **No Foreign Keys** - enables easy merging of databases
- UNIQUE constraints on `endpoint_id` and `(endpoint_id, tag)` pairs
- NOT NULL constraints on critical fields
- Indexes on queryable columns for performance
- Metadata stored as individual columns (not JSON blobs)

### Application Level

- Application-level joins (query separately, combine in code)
- Maximum 25 tags per endpoint (planned for v1.1)
- JSON validation before writing disk files
- Metadata counts updated via application logic
- Endpoint ID: Serial for v1.0 (E001, E002), Random for v1.1 (E12abc)
- File path management (create directories, generate filenames)
- Synchronize SQLite metadata with disk files

---